In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
df = train.merge(identity, how='left', on='TransactionID')

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_full=X_train.copy();
X_test_full=X_test.copy();
y_train_full=y_train.copy();
y_test_full=y_test.copy()
print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")

Train size: (472432, 433)
Test size:  (118108, 433)


In [4]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
num_cols = [col for col in X_train.columns if X_train[col].dtype != 'object']

In [5]:
cols = cat_cols

cat_percent = {
    col: X_train[col].value_counts(normalize=True).mul(100)
    for col in cols
}
cat_group_70_90 = {}
cat_group_90_95 = {}
cat_group_95_plus = {}
for col, dist in cat_percent.items():
    max_percent = dist.max()
    
    if 70 <= max_percent < 90:
       cat_group_70_90[col] = dist
        
    elif 90 <= max_percent < 95:
        cat_group_90_95[col] = dist
        
    elif max_percent >= 95:
        cat_group_95_plus[col] = dist
        

In [6]:
import numpy as np

num_group_70_90 = {}
num_group_90_95 = {}
num_group_95_plus = {}

for col in num_cols:
    data = X_train[col].dropna()
    
    counts, _ = np.histogram(data, bins=20)
    percent = counts / counts.sum() * 100
    
    max_bin = percent.max()
    
    if 70 <= max_bin < 90:
        num_group_70_90[col] = percent
        
    elif 90 <= max_bin < 95:
        num_group_90_95[col] = percent
        
    elif max_bin >= 95:
        num_group_95_plus[col] = percent

In [7]:
id_columns = [col for col in X_train.columns if "id" in col.lower()]
print(id_columns)

['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38']


In [8]:
all_drop_cols = set(
    id_columns +
    list(cat_group_90_95.keys()) +
    list(cat_group_95_plus.keys())
)

X_train = X_train.drop(columns=all_drop_cols, errors="ignore")
print(X_train.shape)
print(all_drop_cols)
X_test = X_test.drop(columns=all_drop_cols, errors="ignore")

(472432, 393)
{'id_09', 'id_03', 'id_23', 'id_15', 'id_17', 'id_24', 'id_35', 'id_06', 'id_33', 'id_07', 'id_11', 'id_30', 'id_02', 'id_01', 'id_38', 'id_16', 'id_04', 'id_29', 'id_19', 'id_28', 'id_36', 'id_20', 'id_12', 'id_13', 'id_37', 'id_08', 'id_22', 'id_27', 'id_25', 'TransactionID', 'id_31', 'id_26', 'id_05', 'id_34', 'id_14', 'id_18', 'id_32', 'id_21', 'M1', 'id_10'}


In [9]:
train_modes = X_train.mode().iloc[0]

X_train = X_train.fillna(train_modes)
X_test = X_test.fillna(train_modes)
X_train_full = X_train_full.fillna(train_modes)
X_test_full = X_test_full.fillna(train_modes)

In [10]:
cat_cols = X_train.select_dtypes(include="object").columns

for col in cat_cols:
    train_mask = X_train[col].str.lower().eq("unknown")
    test_mask = X_test[col].str.lower().eq("unknown")

    X_train.loc[train_mask, col] = train_modes[col]
    X_test.loc[test_mask, col] = train_modes[col]

In [11]:
threshold = 0.99
near_constant_cols = []

for col in X_train.columns:
    top_freq = X_train[col].value_counts(normalize=True, dropna=False).iloc[0]
    
    if top_freq >= threshold:
        near_constant_cols.append(col)

# drop from both train and test
X_train = X_train.drop(columns=near_constant_cols)
X_test = X_test.drop(columns=near_constant_cols)

print("Dropped columns:", near_constant_cols)

Dropped columns: ['addr2', 'C3', 'V1', 'V14', 'V27', 'V28', 'V41', 'V65', 'V68', 'V88', 'V89', 'V107', 'V108', 'V110', 'V111', 'V112', 'V113', 'V114', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V138', 'V141', 'V142', 'V161', 'V162', 'V163', 'V191', 'V192', 'V193', 'V196', 'V240', 'V241', 'V247', 'V248', 'V249', 'V252', 'V253', 'V254', 'V269', 'V305', 'V325', 'V334']


regresion only

In [12]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
skewness = X_train[num_cols].skew().sort_values(ascending=False)

skewed_cols = skewness[skewness > 1].index

# check for non-negative values
valid_cols = [col for col in skewed_cols if (X_train[col] >= 0).all()]

X_train[valid_cols] = np.log1p(X_train[valid_cols])
X_test[valid_cols] = np.log1p(X_test[valid_cols])


regresion only

In [13]:
corr_matrix = X_train.select_dtypes(include=["int64", "float64"]).corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
print(high_corr_cols)
X_train = X_train.drop(columns=high_corr_cols)
X_test = X_test.drop(columns=high_corr_cols)


['C10', 'C11', 'D2', 'V11', 'V16', 'V18', 'V21', 'V22', 'V30', 'V32', 'V33', 'V34', 'V40', 'V42', 'V43', 'V49', 'V52', 'V58', 'V60', 'V63', 'V64', 'V70', 'V72', 'V73', 'V74', 'V79', 'V81', 'V84', 'V85', 'V91', 'V93', 'V94', 'V129', 'V140', 'V145', 'V147', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V165', 'V179', 'V182', 'V183', 'V189', 'V197', 'V198', 'V201', 'V204', 'V206', 'V208', 'V210', 'V213', 'V216', 'V219', 'V222', 'V233', 'V237', 'V239', 'V244', 'V251', 'V255', 'V256', 'V259', 'V265', 'V266', 'V271', 'V272', 'V275', 'V278', 'V289', 'V298', 'V303', 'V304', 'V309', 'V311', 'V313', 'V315', 'V324', 'V330', 'V333', 'V339']


In [14]:
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    rare = freq[freq < 0.01].index
    
    X_train[col] = X_train[col].replace(rare, "Other")
    X_test[col] = X_test[col].replace(rare, "Other")

In [15]:
X_train.shape

(472432, 257)

In [16]:
import category_encoders as ce
woe_encoder = ce.WOEEncoder(cols=X_train.select_dtypes(include=['object']).columns)
X_train_encoded = woe_encoder.fit_transform(X_train, y_train)
X_test_encoded = woe_encoder.transform(X_test)
print(X_train_encoded.head())

        TransactionDT  TransactionAmt  ProductCD  card1  card2     card3  \
5307           174911        4.094345  -0.547281   4988  334.0  5.017280   
191582        4301977        3.193681   1.286106   3867  296.0  5.225747   
260168        6229929        3.609566  -0.547281  12577  268.0  5.017280   
18516          497176        4.615121   0.071001   6019  583.0  5.017280   
47538         1124702        4.330733   0.327350  16075  514.0  5.017280   

           card4  card5     card6  addr1  ...  V328  V329  V331  V332  V335  \
5307   -0.007670  226.0 -0.379514  315.0  ...   0.0   0.0   0.0   0.0   0.0   
191582 -0.007670  226.0  0.682442  299.0  ...   0.0   0.0   0.0   0.0   0.0   
260168 -0.007670  166.0 -0.379514  476.0  ...   0.0   0.0   0.0   0.0   0.0   
18516  -0.007670  226.0  0.682442  126.0  ...   0.0   0.0   0.0   0.0   0.0   
47538  -0.020975  102.0  0.682442  325.0  ...   0.0   0.0   0.0   0.0   0.0   

        V336  V337  V338  DeviceType  DeviceInfo  
5307     0.0   0.

In [17]:
import category_encoders as ce

woe_encoder_full = ce.WOEEncoder(
    cols=X_train_full.select_dtypes(include=['object']).columns
)

X_train_encoded_full = woe_encoder_full.fit_transform(X_train_full, y_train_full)
X_test_encoded_full = woe_encoder_full.transform(X_test_full)

print(X_train_encoded_full.head())

        TransactionID  TransactionDT  TransactionAmt  ProductCD  card1  card2  \
5307          2992307         174911          59.000  -0.547281   4988  334.0   
191582        3178582        4301977          23.378   1.286106   3867  296.0   
260168        3247168        6229929          35.950  -0.547281  12577  268.0   
18516         3005516         497176         100.000   0.071001   6019  583.0   
47538         3034538        1124702          75.000   0.327350  16075  514.0   

        card3     card4  card5     card6  ...     id_31  id_32     id_33  \
5307    150.0 -0.007670  226.0 -0.379514  ... -0.514399    NaN -0.045912   
191582  185.0 -0.007670  226.0  0.682442  ...  0.704913    NaN -0.045912   
260168  150.0 -0.007670  166.0 -0.379514  ... -0.514399    NaN -0.045912   
18516   150.0 -0.007670  226.0  0.682442  ...  0.188526   24.0 -0.081097   
47538   150.0 -0.020975  102.0  0.682442  ...  0.854557   32.0  0.580520   

           id_34     id_35     id_36     id_37     id_38

In [18]:
print(X_train_encoded_full.select_dtypes(include="object").columns)
print(X_test_encoded_full.select_dtypes(include="object").columns)
X_train_encoded_full.dtypes

Index([], dtype='object')
Index([], dtype='object')


TransactionID       int64
TransactionDT       int64
TransactionAmt    float64
ProductCD         float64
card1               int64
                   ...   
id_36             float64
id_37             float64
id_38             float64
DeviceType        float64
DeviceInfo        float64
Length: 433, dtype: object

In [19]:
print(X_train_encoded.select_dtypes(include="object").columns)
print(X_test_encoded.select_dtypes(include="object").columns)
X_train_encoded.dtypes

Index([], dtype='object')
Index([], dtype='object')


TransactionDT       int64
TransactionAmt    float64
ProductCD         float64
card1               int64
card2             float64
                   ...   
V336              float64
V337              float64
V338              float64
DeviceType        float64
DeviceInfo        float64
Length: 257, dtype: object

In [20]:
X_train_encoded_full = X_train_encoded_full.fillna(0)
X_test_encoded_full = X_test_encoded_full.fillna(0)
X_train_encoded = X_train_encoded_full.fillna(0)
X_test_encoded = X_test_encoded_full.fillna(0)

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Model
model = LogisticRegression(max_iter=1000)

# -------------------
# FULL FEATURES
# -------------------
model.fit(X_train_encoded_full, y_train)
pred_full = model.predict_proba(X_test_encoded_full)[:, 1]
auc_full = roc_auc_score(y_test, pred_full)

# -------------------
# REDUCED FEATURES
# -------------------
model.fit(X_train_encoded, y_train)
pred_reduced = model.predict_proba(X_test_encoded)[:, 1]
auc_reduced = roc_auc_score(y_test, pred_reduced)

print("AUC full:", auc_full)
print("AUC reduced:", auc_reduced)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

AUC full: 0.7526010250367245
AUC reduced: 0.7526010250367245


In [22]:
pred = model.predict(X_test_encoded)
pred_full = model.predict(X_test_encoded_full)

from sklearn.metrics import recall_score
recall = recall_score(y_test, pred)
recall_full = recall_score(y_test_full, pred_full)

print(recall)
print("full",recall)

0.0054219707685054215
full 0.0054219707685054215


In [23]:
y_train.value_counts()

isFraud
0    456011
1     16421
Name: count, dtype: int64